# Milestone 3 — Model Development & Training

This notebook trains five supervised classifiers on both datasets:
1. **Logistic Regression** — baseline interpretable model
2. **Decision Tree** — rule-based nonlinear classifier
3. **Random Forest** — ensemble, reduces DT overfitting
4. **Support Vector Machine (SVM)** — effective in high-dimensional space
5. **XGBoost** — gradient boosting, state-of-the-art benchmark

Each model is:
- Trained on the SMOTE-balanced training set
- Tuned via GridSearchCV (3-fold CV on training data)
- Validated on the held-out validation set
- Saved to `models/`

In [ ]:
import pandas as pd
import numpy as np
import pickle
import time
import warnings
import os

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)

warnings.filterwarnings('ignore')
MODELS  = '../models'
SEED = 42

## 1. Load Processed Data

In [ ]:
def load_dataset(prefix, path='../data/processed'):
    X_train = pd.read_csv(f'{path}/{prefix}_X_train.csv')
    X_val   = pd.read_csv(f'{path}/{prefix}_X_val.csv')
    X_test  = pd.read_csv(f'{path}/{prefix}_X_test.csv')
    y_train = pd.read_csv(f'{path}/{prefix}_y_train.csv').values.ravel()
    y_val   = pd.read_csv(f'{path}/{prefix}_y_val.csv').values.ravel()
    y_test  = pd.read_csv(f'{path}/{prefix}_y_test.csv').values.ravel()
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train_t, X_val_t, X_test_t, y_train_t, y_val_t, y_test_t = load_dataset('telco')
X_train_i, X_val_i, X_test_i, y_train_i, y_val_i, y_test_i = load_dataset('india')

print('Telco  — Train:', X_train_t.shape, '| Val:', X_val_t.shape, '| Test:', X_test_t.shape)
print('India  — Train:', X_train_i.shape, '| Val:', X_val_i.shape, '| Test:', X_test_i.shape)

## 2. Evaluation Helper

In [ ]:
def evaluate(model, X, y, split_name='Val'):
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1] if hasattr(model, 'predict_proba') else None
    metrics = {
        'Split'     : split_name,
        'Accuracy'  : round(accuracy_score(y, y_pred), 4),
        'Precision' : round(precision_score(y, y_pred, zero_division=0), 4),
        'Recall'    : round(recall_score(y, y_pred, zero_division=0), 4),
        'F1-Score'  : round(f1_score(y, y_pred, zero_division=0), 4),
        'ROC-AUC'   : round(roc_auc_score(y, y_prob), 4) if y_prob is not None else None
    }
    return metrics

## 3. Model Definitions & Hyperparameter Grids

In [ ]:
models_config = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=SEED),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'solver': ['lbfgs', 'liblinear']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=SEED),
        'params': {
            'max_depth': [4, 6, 8, 12, None],
            'min_samples_split': [2, 5, 10],
            'criterion': ['gini', 'entropy']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=SEED, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [6, 10, None],
            'min_samples_split': [2, 5]
        }
    },
    # SVM: narrow grid — RBF kernel only, scale gamma fixed; linear SVM is already
    # covered by Logistic Regression. Keeps GridSearchCV tractable on large datasets.
    'SVM': {
        'model': SVC(probability=True, kernel='rbf', gamma='scale', random_state=SEED),
        'params': {
            'C': [0.1, 1, 10],
        }
    },
    'XGBoost': {
        'model': XGBClassifier(eval_metric='logloss', random_state=SEED,
                               use_label_encoder=False, n_jobs=-1),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [3, 5, 7],
            'learning_rate': [0.05, 0.1, 0.2]
        }
    }
}

print('Models configured:', list(models_config.keys()))

## 4. Training Function

In [ ]:
def train_all_models(X_train, y_train, X_val, y_val, dataset_label):
    results = []
    trained = {}
    
    for name, cfg in models_config.items():
        print(f'  [{dataset_label}] Training {name}...', end=' ')
        t0 = time.time()
        
        gs = GridSearchCV(
            cfg['model'], cfg['params'],
            cv=3, scoring='f1', n_jobs=-1, refit=True
        )
        gs.fit(X_train, y_train)
        best = gs.best_estimator_
        elapsed = round(time.time() - t0, 1)
        
        val_metrics = evaluate(best, X_val, y_val, 'Validation')
        val_metrics['Model']      = name
        val_metrics['Dataset']    = dataset_label
        val_metrics['Best Params']= str(gs.best_params_)
        val_metrics['Train Time'] = f'{elapsed}s'
        results.append(val_metrics)
        trained[name] = best
        
        print(f'done in {elapsed}s | Val F1={val_metrics["F1-Score"]:.4f} | AUC={val_metrics["ROC-AUC"]}')
    
    return pd.DataFrame(results), trained

## 5. Train on Telco Dataset

In [ ]:
print('=== Training on TELCO dataset ===')
telco_results, telco_models = train_all_models(
    X_train_t, y_train_t, X_val_t, y_val_t, 'Telco')

display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Train Time']
print('\n=== Telco Validation Results ===')
display(telco_results[display_cols].sort_values('F1-Score', ascending=False).reset_index(drop=True))

## 6. Train on India Dataset

In [ ]:
print('=== Training on INDIA dataset ===')
india_results, india_models = train_all_models(
    X_train_i, y_train_i, X_val_i, y_val_i, 'India')

print('\n=== India Validation Results ===')
display(india_results[display_cols].sort_values('F1-Score', ascending=False).reset_index(drop=True))

## 7. Save Trained Models

In [ ]:
for name, model in telco_models.items():
    fname = name.lower().replace(' ', '_')
    with open(f'{MODELS}/telco_{fname}.pkl', 'wb') as f:
        pickle.dump(model, f)

for name, model in india_models.items():
    fname = name.lower().replace(' ', '_')
    with open(f'{MODELS}/india_{fname}.pkl', 'wb') as f:
        pickle.dump(model, f)

# Save validation results for notebook 04
all_results = pd.concat([telco_results, india_results], ignore_index=True)
all_results.to_csv(f'{MODELS}/validation_results.csv', index=False)

print('All models saved to:', MODELS)
for f in sorted(os.listdir(MODELS)):
    if f.endswith('.pkl'):
        print(' ', f)

In [ ]:
print('Milestone 3 — Model Training COMPLETE')